# This should run in local environment, does not work in Colab

In [1]:
%pip install TikTokApi
%pip install nest_asyncio
%pip install pandas
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
!python3 -m playwright install webkit

In [4]:
import subprocess
result = subprocess.run(
    ["python3", "-m", "playwright", "install", "webkit"], 
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

In [5]:
import asyncio
import os
from TikTokApi import TikTokApi
import nest_asyncio
nest_asyncio.apply()

/Users/yunangao/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [13]:
import nest_asyncio
nest_asyncio.apply()
from TikTokApi import TikTokApi
import asyncio

ms_token = "-F83TZDErbB_LlXjV61qyJWbv6vsFzoUJ_Td0pIaHeUyq7x-0X_z7wpCKe8VvIScUcYYb0HIyxHiCnCBLfOYLzbyA3S5OlWVjN7Fnw_-Tdgwyv0a8W0RpsSzleCx3TWR2oOId-W3654C7RV5UbdTCJo="
video_url = "https://www.tiktok.com/@therock/video/7248300636498890011"

async def get_comments_v2():
    async with TikTokApi() as api:
        await api.create_sessions(
            ms_tokens=[ms_token],
            num_sessions=1,
            sleep_after=10,
            headless=False,
            browser="webkit",
            starting_url="https://www.tiktok.com"
        )
        # Use URL instead of id
        video = api.video(url=video_url)
        await video.info()
        
        count = 0
        async for comment in video.comments(count=30):
            print(comment.as_dict)
            count += 1
        print(f"\n✅ Total: {count} comments")

await get_comments_v2()

EmptyResponseException: None -> TikTok returned an empty response. They are detecting you're a bot, try some of these: headless=False, browser='webkit', consider using a proxy

In [6]:
ms_token = "EUdvZNYeoNQ6YNY9Ug20W7VceJLonUZpLXvUQKuPKhrXlzEmSJ2y0Q10T_B7nWLX7KIJPOsbN6NC49vpEkGar_DKSjPt8GV35VD5lWVEjCwW_c9COXZctTBh8PBG0-QxnWZMQPXn9n7FiTN7BYv3_90="

In [7]:
# trending videos under #beauty

import nest_asyncio
nest_asyncio.apply()
from TikTokApi import TikTokApi
import asyncio


async def get_hashtag_videos():
    try:
        async with TikTokApi() as api:
            await api.create_sessions(
                ms_tokens=[ms_token],
                num_sessions=1,
                sleep_after=10,
                headless=False,
                browser="webkit"  
            )
            tag = api.hashtag(name="beauty")  
            async for video in tag.videos(count=10):
                desc = video.as_dict.get('desc', '')[:40]
                plays = video.as_dict.get('stats', {}).get('playCount', 0)
                print(f"[Description]: {desc} | [Views]: {plays:,}")
    except Exception as e:
        print(f"[error]: {e}")

await get_hashtag_videos() 

[Description]: Here are some of mine and my mom’s favor | [Views]: 2,200,000
[Description]: Twist, Swipe, and Glow! ✨ Your Glow is o | [Views]: 2,200,000
[Description]: rating all of my skin tints 🤭 #honestrev | [Views]: 3,400,000
[Description]: It’s time for an updated makeup routine! | [Views]: 8,200,000
[Description]: What color should I do next?! 💄😍 #makeup | [Views]: 12,000,000
[Description]: Telling the story of “The Beauty” from B | [Views]: 8,000,000
[Description]: she’s a star🍓💫  #bodyoil #bodycare #skin | [Views]: 1,900,000
[Description]: @meritbeauty is such a popular minimalis | [Views]: 1,600,000
[Description]: how to get rid of the winter uglies 101  | [Views]: 858,700
[Description]: Part 2 The PEEL! You guys. It makes the  | [Views]: 14,600,000
[Description]: Go follow the queen @Alieenor 🧡 for more | [Views]: 2,900,000
[Description]: here's 5 makeup tips to help your makeup | [Views]: 1,500,000
[Description]: 6 Classic Hollywood Beauties Then and No | [Views]: 24,700,00

In [ ]:
# Scrape videos based on hashtags, and label them based on follower count (creator tiers), content type, post time, etc. Calculate calculated fields
# such as engagement rate

import nest_asyncio
nest_asyncio.apply()
from TikTokApi import TikTokApi
import asyncio
import pandas as pd
from datetime import datetime
import os

ms_token = "N90DarJlhkpkGch8xkSB_GvtaQ2CfS-ZvFHLqXSP-MiGSlQGbnA8vGxz50wV5hM_S7pZocigBSqY-PaiR-rFixFBJSAbUJVrkSHbw5LK5_oJS6ruWH4v3oIu40cQcbgemcT76PCtKKgTjaG4WBdmm5g="

async def get_beauty_data():
    videos_data = []
    
    hashtags = [
        # General beauty
        "beauty", "skincare", "makeup", "skincareroutine", "makeuptutorial",
        # Content types
        "grwm", "makeupreview", "skincarehaul", "beautytips", "skincaretips",
        # Shopping related
        "sephorahaul", "ultabeauty", "drugstorebeauty", "amazonbeauty", "beautyhaul",
        # Product specific
        "cleanbeauty", "skintok", "serumreview", "moisturizer", "sunscreen",
        # TikTok Shop related
        "tiktokmademebuyit", "beautytok", "glowup", "antiaging", "acneskincare",
        # Brand names
        "cetaphil", "cerave", "laneige", "rarebeauty", "elfcosmetics",
        # Skin concerns
        "oilyskin", "dryskin", "sensitiveskin", "hyperpigmentation", "skincarejunkie",
        # Korean beauty
        "kbeauty", "koreanskincare", "glasskin", "doublecleanse", "essencetoner",
        # Trending
        "beautyhacks", "skincarehacks", "makeupdupes", "affordablebeauty", "beautyunder20"
    ]
    
    try:
        async with TikTokApi() as api:
            await api.create_sessions(
                ms_tokens=[ms_token],
                num_sessions=1,
                sleep_after=10,
                headless=False,
                browser="webkit"
            )
            
            for i, tag_name in enumerate(hashtags):
                print(f"[{i+1}/{len(hashtags)}] Scraping #{tag_name}...")
                try:
                    tag = api.hashtag(name=tag_name)
                    video_count = 0
                    
                    async for video in tag.videos(count=50):
                        d = video.as_dict
                        
                        # Creator tier classification
                        follower_count = d.get("authorStats", {}).get("followerCount", 0)
                        if follower_count >= 1000000:
                            creator_tier = "mega"
                        elif follower_count >= 100000:
                            creator_tier = "macro"
                        elif follower_count >= 10000:
                            creator_tier = "micro"
                        else:
                            creator_tier = "nano"
                        
                        # Content type classification
                        desc = d.get("desc", "").lower()
                        if any(w in desc for w in ["tutorial", "how to", "step", "teach"]):
                            content_type = "tutorial"
                        elif any(w in desc for w in ["review", "honest", "worth it", "test"]):
                            content_type = "review"
                        elif any(w in desc for w in ["haul", "bought", "shopping"]):
                            content_type = "haul"
                        elif any(w in desc for w in ["grwm", "get ready"]):
                            content_type = "grwm"
                        elif any(w in desc for w in ["routine", "everyday", "daily"]):
                            content_type = "routine"
                        else:
                            content_type = "other"
                        
                        # Post time breakdown
                        create_time = d.get("createTime", 0)
                        try:
                            dt = datetime.fromtimestamp(create_time)
                            post_hour  = dt.hour
                            post_day   = dt.strftime("%A")
                            post_month = dt.month
                        except:
                            post_hour  = None
                            post_day   = None
                            post_month = None
                        
                        videos_data.append({
                            # Source
                            "source_hashtag":     tag_name,
                            # Content
                            "video_id":           d.get("id", ""),
                            "desc":               d.get("desc", ""),
                            "create_time":        create_time,
                            "post_hour":          post_hour,
                            "post_day":           post_day,
                            "post_month":         post_month,
                            "duration":           d.get("video", {}).get("duration", 0),
                            "music_title":        d.get("music", {}).get("title", ""),
                            "is_commerce_music":  d.get("music", {}).get("is_commerce_music", False),
                            "hashtags":           [t.get("title","") for t in d.get("challenges", [])],
                            "content_type":       content_type,
                            # Creator
                            "author":             d.get("author", {}).get("uniqueId", ""),
                            "verified":           d.get("author", {}).get("verified", False),
                            "follower_count":     follower_count,
                            "following_count":    d.get("authorStats", {}).get("followingCount", 0),
                            "total_likes":        d.get("authorStats", {}).get("heartCount", 0),
                            "video_count":        d.get("authorStats", {}).get("videoCount", 0),
                            "creator_tier":       creator_tier,
                            # Engagement
                            "play_count":         d.get("stats", {}).get("playCount", 0),
                            "like_count":         d.get("stats", {}).get("diggCount", 0),
                            "comment_count":      d.get("stats", {}).get("commentCount", 0),
                            "share_count":        d.get("stats", {}).get("shareCount", 0),
                            "collect_count":      d.get("stats", {}).get("collectCount", 0),
                            "is_ad":              d.get("isAd", False),
                        })
                        video_count += 1
                    
                    print(f"Got {video_count} videos from #{tag_name}")
                    
                except Exception as e:
                    print(f"#{tag_name} failed: {e}")
                    continue
        
        # Build dataframe
        df = pd.DataFrame(videos_data)
        
        # Remove duplicates
        before = len(df)
        df = df.drop_duplicates(subset="video_id")
        after = len(df)
        print(f"\nRemoved {before - after} duplicates")
        
        # Calculate engagement rate
        df["engagement_rate"] = (
            (df["like_count"] + df["comment_count"] + df["share_count"] + df["collect_count"])
            / df["play_count"].replace(0, 1)
        ).round(4)
        
        # Summary
        print(f"\nTotal videos: {len(df)}")
        print(f"Unique creators: {df['author'].nunique()}")
        print(f"\nCreator tier breakdown:")
        print(df['creator_tier'].value_counts())
        print(f"\nContent type breakdown:")
        print(df['content_type'].value_counts())
        
        # Save with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        filename = f"/Users/yunangao/Desktop/tiktok_beauty_{timestamp}.csv"
        df.to_csv(filename, index=False)
        print(f"\nSaved to Desktop: tiktok_beauty_{timestamp}.csv")
        
        return df
    
    except Exception as e:
        print(f"[Error]: {e}")
        return pd.DataFrame()

df = await get_beauty_data()

[1/45] Scraping #beauty...
Got 59 videos from #beauty
[2/45] Scraping #skincare...
Got 60 videos from #skincare
[3/45] Scraping #makeup...
Got 60 videos from #makeup
[4/45] Scraping #skincareroutine...
Got 59 videos from #skincareroutine
[5/45] Scraping #makeuptutorial...
Got 60 videos from #makeuptutorial
[6/45] Scraping #grwm...
Got 60 videos from #grwm
[7/45] Scraping #makeupreview...
Got 60 videos from #makeupreview
[8/45] Scraping #skincarehaul...
Got 59 videos from #skincarehaul
[9/45] Scraping #beautytips...
Got 59 videos from #beautytips
[10/45] Scraping #skincaretips...
Got 60 videos from #skincaretips
[11/45] Scraping #sephorahaul...
Got 60 videos from #sephorahaul
[12/45] Scraping #ultabeauty...
Got 60 videos from #ultabeauty
[13/45] Scraping #drugstorebeauty...
Got 60 videos from #drugstorebeauty
[14/45] Scraping #amazonbeauty...
Got 60 videos from #amazonbeauty
[15/45] Scraping #beautyhaul...
Got 60 videos from #beautyhaul
[16/45] Scraping #cleanbeauty...
Got 60 videos fro

In [9]:
import glob
# we run the above code multiple times to get more data by using different ms_tokens and remove duplicates
# we then combine all the csv files we have collected so far
all_files = glob.glob("/Users/yunangao/Desktop/tiktok_beauty_*.csv")
print(f"Found {len(all_files)} files")

df_combined = pd.concat([pd.read_csv(f) for f in all_files])
df_combined = df_combined.drop_duplicates(subset="video_id")
print(f"Total unique videos: {len(df_combined)}")

df_combined.to_csv("/Users/yunangao/Desktop/tiktok_beauty_final.csv", index=False)
print("Saved: tiktok_beauty_final.csv")

Found 1 files
Total unique videos: 2361
Saved: tiktok_beauty_final.csv
